In [1]:
import os
import json

import pandas as pd

In [3]:
v1_pth = "database"
v2_pth = "../../../data/ObjectDetectionV3"

with open(os.path.join(v1_pth, "train_test_split.json"), "rt") as f:
    v1_train_test_split = json.load(f)

with open(os.path.join(v2_pth, "train_test_split.json"), "rt") as f:
    v2_train_test_split = json.load(f)

In [4]:
v1_df = pd.read_csv(os.path.join(v1_pth, "images_grouped.csv"))
v2_df = pd.read_csv(os.path.join(v2_pth, "images_grouped.csv"))

v1_df.shape, v2_df.shape

((1032, 7), (1457, 7))

In [5]:
v1_df.head()

,id,name,width,height,batch,folder,group_id
0,bdd640fb-0667-4ad1-9c80-317fa3b1799d,Cam3_H243_discount_14.png,640.0,480.0,batch1_10.5,batch1,120287d6-9abf-49b8-ac00-72c72f9e383b
1,fc377a4c-4a15-444d-85e7-ce8a3a578a8e,Cam3_H243_discount_15.png,640.0,480.0,batch1_10.5,batch1,120287d6-9abf-49b8-ac00-72c72f9e383b
2,ff50bde4-3825-47b8-9cab-cc97663f1c97,Cam3_H243_discount_16.png,640.0,480.0,batch1_10.5,batch1,120287d6-9abf-49b8-ac00-72c72f9e383b
3,5cec4eb5-edd9-4831-9ca3-5cfb04fc6d82,Cam3_H243_discount_17.png,640.0,480.0,batch1_10.5,batch1,120287d6-9abf-49b8-ac00-72c72f9e383b
4,b92da22b-21df-406f-8a0b-3c3336d8393a,Cam3_H243_discount_18.png,640.0,480.0,batch1_10.5,batch1,120287d6-9abf-49b8-ac00-72c72f9e383b


In [6]:
v1_df["path"] = v1_df.apply(lambda row: os.path.join(row["batch"], row["folder"], row["name"]), axis=1)
v2_df["path"] = v2_df.apply(lambda row: os.path.join(row["batch"], row["folder"], row["name"]), axis=1)

In [7]:
df = pd.merge(
    v1_df,
    v2_df,
    on="path",
    how="inner",
    indicator=True
)
df.head()

,id_x,name_x,width_x,height_x,batch_x,folder_x,group_id_x,path,id_y,name_y,width_y,height_y,batch_y,folder_y,group_id_y,_merge
0,bdd640fb-0667-4ad1-9c80-317fa3b1799d,Cam3_H243_discount_14.png,640.0,480.0,batch1_10.5,batch1,120287d6-9abf-49b8-ac00-72c72f9e383b,batch1_10.5/batch1/Cam3_H243_discount_14.png,bdd640fb-0667-4ad1-9c80-317fa3b1799d,Cam3_H243_discount_14.png,640.0,480.0,batch1_10.5,batch1,5204194a-5097-42a7-b81a-94791a1e482a,both
1,fc377a4c-4a15-444d-85e7-ce8a3a578a8e,Cam3_H243_discount_15.png,640.0,480.0,batch1_10.5,batch1,120287d6-9abf-49b8-ac00-72c72f9e383b,batch1_10.5/batch1/Cam3_H243_discount_15.png,fc377a4c-4a15-444d-85e7-ce8a3a578a8e,Cam3_H243_discount_15.png,640.0,480.0,batch1_10.5,batch1,5204194a-5097-42a7-b81a-94791a1e482a,both
2,ff50bde4-3825-47b8-9cab-cc97663f1c97,Cam3_H243_discount_16.png,640.0,480.0,batch1_10.5,batch1,120287d6-9abf-49b8-ac00-72c72f9e383b,batch1_10.5/batch1/Cam3_H243_discount_16.png,ff50bde4-3825-47b8-9cab-cc97663f1c97,Cam3_H243_discount_16.png,640.0,480.0,batch1_10.5,batch1,5204194a-5097-42a7-b81a-94791a1e482a,both
3,5cec4eb5-edd9-4831-9ca3-5cfb04fc6d82,Cam3_H243_discount_17.png,640.0,480.0,batch1_10.5,batch1,120287d6-9abf-49b8-ac00-72c72f9e383b,batch1_10.5/batch1/Cam3_H243_discount_17.png,5cec4eb5-edd9-4831-9ca3-5cfb04fc6d82,Cam3_H243_discount_17.png,640.0,480.0,batch1_10.5,batch1,5204194a-5097-42a7-b81a-94791a1e482a,both
4,b92da22b-21df-406f-8a0b-3c3336d8393a,Cam3_H243_discount_18.png,640.0,480.0,batch1_10.5,batch1,120287d6-9abf-49b8-ac00-72c72f9e383b,batch1_10.5/batch1/Cam3_H243_discount_18.png,b92da22b-21df-406f-8a0b-3c3336d8393a,Cam3_H243_discount_18.png,640.0,480.0,batch1_10.5,batch1,5204194a-5097-42a7-b81a-94791a1e482a,both


In [8]:
sum(df["id_x"] == df["id_y"]), sum(df["group_id_x"] == df["group_id_y"])

(980, 0)

In [9]:
val_df = df[df["id_x"].isin(v1_train_test_split["test"])]
sum(val_df["id_x"] == val_df["id_y"]) == val_df.shape[0]

True

In [16]:
len(v1_train_test_split["test"]), \
    len(set(v1_train_test_split["test"]).intersection(v2_train_test_split["test"])), \
    len(set(v1_train_test_split["test"]).intersection(v2_train_test_split["train"]))

(165, 39, 126)

In [17]:
len(v2_train_test_split["test"]), \
    len(set(v2_train_test_split["test"]).intersection(v1_train_test_split["test"])), \
    len(set(v2_train_test_split["test"]).intersection(v1_train_test_split["train"]))

(292, 39, 205)

In [23]:
val_set = set(v1_train_test_split["test"] + v2_train_test_split["test"])\
    .difference(v1_train_test_split["train"]).difference(v2_train_test_split["train"])

len(val_set)

87